# 差分メモ

- 対象: `[2-1]dpc-starter-train-v1.ipynb`
- 元: `[2]dpc-starter-train-v1.ipynb`
- 種別: 改修（学習用: full-train）
- 変更点:
  - エポック数を20 → 25
- 変更理由/仮説:
  - エポック数ではまだ学習が完了していないように思われるため


# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [ ]:
# (Optional) Kaggle environment usually includes required libs.
# If your environment is missing something, uncomment and install as needed.
# !pip install -q datasets transformers accelerate


In [ ]:
import os
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)


In [ ]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is a strong choice.
    MODEL_NAME = "google/byt5-base"

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    # Training
    EPOCHS = 25
    LEARNING_RATE = 2e-4

    # ByT5-base is heavier; keep per-device batch small and use grad-accum.
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8

    # Output
    OUTPUT_DIR = "./byt5-akkadian-model/fulltrain_byt5-base_multitask"


In [ ]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [ ]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [ ]:
print(f"Original Train Data: {len(train_df)} docs")

In [ ]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [ ]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

train_expanded = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(train_expanded.columns)

# ------------------------------------------
# Prefix multi-task: akk→en + en→akk (2x)
# ------------------------------------------
multitask_df = pd.concat(
    [
        train_expanded.assign(
            task="akk2en",
            source=train_expanded["transliteration"],
            target=train_expanded["translation"],
        ),
        train_expanded.assign(
            task="en2akk",
            source=train_expanded["translation"],
            target=train_expanded["transliteration"],
        ),
    ],
    ignore_index=True,
)[["oare_id", "task", "source", "target"]]

print(f"Multi-task Train Data: {len(multitask_df)} samples (2x directions)")
# Deterministic shuffle so batches mix directions
multitask_df = multitask_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(multitask_df.head())


In [ ]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Unify common gap markers to a single token for better pattern consistency.
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_function(examples):
    tasks = examples["task"]
    sources = examples["source"]
    targets = examples["target"]

    inputs = []
    out_targets = []

    for t, s, y in zip(tasks, sources, targets):
        if t == "akk2en":
            s_norm = normalize_transliteration(s)
            inputs.append(PREFIX_AKK_EN + s_norm)
            out_targets.append(str(y))
        elif t == "en2akk":
            inputs.append(PREFIX_EN_AKK + str(s))
            out_targets.append(normalize_transliteration(y))
        else:
            raise ValueError(f"Unknown task: {t}")

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(out_targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
# ==========================================
# 4. Full-data training (fine-tuning)
# ==========================================

# Build a single HF dataset (full train, no CV).
ds_train = Dataset.from_pandas(multitask_df)

# Tokenize
train_tokenized = ds_train.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_train.column_names,
)

# Model
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,

    # No CV / no validation in this notebook
    eval_strategy="no",

    save_strategy="no",

    # Checkpoints (incl. optimizer states) can be huge; save only final model at the end.

    learning_rate=Config.LEARNING_RATE,
    fp16=False,

    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,

    weight_decay=0.01,
    num_train_epochs=Config.EPOCHS,

    predict_with_generate=False,

    logging_strategy="steps",
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting Training (full train, multi-task, FP32 mode)...")
trainer.train()

trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Saved model to: {Config.OUTPUT_DIR}")


In [ ]:
# --- Notes ---
# This notebook trains a single model using all training data (no CV).
# Output:
#   ./byt5-akkadian-model/fulltrain_byt5-base_multitask/
